### 1. ✅ Retry with Exponential Backoff — (AWS/API calls context)
**Why it impresses**: parametrized decorator factory, real AWS/API relevance, exponential backoff is a known distributed systems pattern.

In [ ]:
import functools
import time
import logging

logger = logging.getLogger(__name__)

# params: number of retries, exceptions list, backoff factor
# in wrapper:
    # set delay
    # retry logic in a loop: 1) happy path; 2) exception & retry
    # handle exceptions raise by original func:
        # 1) if max attempts is reached - log, re-raise
        # 2) otherwise, log, sleep, reset delay - keep looping
def retry(max_attempts=3, exceptions=(Exception,), backoff_factor=2):
    """
    Decorator factory for retrying flaky operations.
    Real use case: calling AWS S3, external APIs, DB connections.
    """
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            delay = 1
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)       # 1) happy path
                except exceptions as e:                # 2) exception & retry
                    if attempt == max_attempts:
                        logger.error(
                            f"[{func.__name__}] Failed after "
                            f"{max_attempts} attempts. Error: {e}"
                        )
                        raise  # throws the SAME exception up the call stack
                    logger.warning(
                        f"[{func.__name__}] Attempt {attempt} failed: {e}. "
                        f"Retrying in {delay}s..."
                    )
                    time.sleep(delay)
                    delay *= backoff_factor  # exponential backoff
        return wrapper
    return decorator

In [ ]:
# --- Usage ---
import requests

@retry(max_attempts=4, exceptions=(requests.Timeout, requests.ConnectionError))
def fetch_user_data(user_id: int):
    response = requests.get(
        f"https://api.example.com/users/{user_id}",
        timeout=3
    )
    response.raise_for_status()   # raised HTTPError - is a sibling of Timeout & ConnectionError
    return response.json()

**The API We'll Use: `httpbin.org`**
Perfect for debugging — it lets you **trigger any HTTP status code on demand**, simulate delays, and inspect requests. No auth needed.
```shell
GET https://httpbin.org/status/500  → forces a 500 error
GET https://httpbin.org/status/200  → success
GET https://httpbin.org/delay/5     → delays 5s (triggers Timeout)
GET https://httpbin.org/get         → simple successful GET
```

**Step 1 — The Exception Hierarchy You Must Know**
```shell
BaseException
 └── Exception
      └── IOError
           └── OSError
                └── requests.exceptions.RequestException   ← base for ALL requests errors
                     ├── requests.exceptions.ConnectionError  ← no network / DNS fail
                     ├── requests.exceptions.Timeout          ← took too long
                     ├── requests.exceptions.HTTPError        ← raise_for_status() raises THIS
                     └── requests.exceptions.URLRequired
```
This hierarchy matters because when you write `except (Timeout, ConnectionError)` you are **not** catching `HTTPError` — they are siblings, not parent/child.

---

**Step 2 — What `raise_for_status()` Actually Does**

In [ ]:
import requests

response = requests.get("https://httpbin.org/status/500")

# At this point — NO exception yet. requests got a response.
# response.status_code = 500
# response.ok          = False
# The HTTP conversation completed — it's just a "bad" status

print(response.status_code)  # 500  ← no error yet!

# NOW the exception is raised
response.raise_for_status()  # raises requests.exceptions.HTTPError

# Equivalent to what raise_for_status does internally:
if response.status_code >= 400:
    raise requests.exceptions.HTTPError(
        f"400 Client Error / 500 Server Error",
        response=response   # ← attaches the response object to the exception!
    )

```shell
Timeline:

requests.get() ──→ TCP connect ──→ HTTP request ──→ receives response ──→ returns Response object
                                                                                    │
                                                                          No exception yet!
                                                                                    │
                                                              response.raise_for_status()
                                                                                    │
                                                              ┌─────────────────────┘
                                                              │
                                                   status >= 400?
                                                   YES → raises HTTPError
                                                   NO  → does nothing, returns None
```

---

**Step 3 — What Re-raise Does (`raise` with no argument)**

In [ ]:
try:
    risky_operation()
except Exception as e:
    print(f"Logging: {e}")
    raise          # ← re-raise: throws the SAME exception up the call stack
    # vs
    raise e        # ← also re-raises but RESETS the traceback (lose context!)
    # vs
    raise ValueError("new error") from e  # ← chained exception, wraps original

```shell
Call Stack when re-raise happens:

main()
  └── fetch_user_data(user_id=1)        ← your function
        └── wrapper(*args, **kwargs)    ← the decorator
              └── func(*args, **kwargs) ← original function call
                    └── raise_for_status() raises HTTPError
                          ↑
              except HTTPError as e: ── attempt < max → log + sleep + retry
                                     ── attempt == max → RAISE (re-raise up to caller)
                          ↑
        caller of fetch_user_data receives the original HTTPError
        with full traceback intact
```

---

### Full Working Debuggable Example

In [4]:
import functools
import time
import logging
import requests
from requests.exceptions import Timeout, ConnectionError, HTTPError

# ── Logging setup ──────────────────────────────────────────────
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)


# ── Retry Decorator ────────────────────────────────────────────
def retry(max_attempts=3, exceptions=(Exception,), backoff_factor=2):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            delay = 1

            for attempt in range(1, max_attempts + 1):
                try:
                    # ← BREAKPOINT A: about to call the real function
                    logger.debug(
                        f"[{func.__name__}] Attempt {attempt}/{max_attempts}"
                    )
                    result = func(*args, **kwargs)
                    # ← BREAKPOINT B: function succeeded
                    logger.debug(f"[{func.__name__}] Success on attempt {attempt}")
                    return result

                except exceptions as e:
                    # ← BREAKPOINT C: an expected exception was caught here
                    # e            → the actual exception object
                    # type(e)      → its class (Timeout, HTTPError, etc.)
                    # e.response   → the Response object (only on HTTPError)
                    # e.args       → raw args passed to the exception
                    logger.warning(
                        f"[{func.__name__}] Attempt {attempt} caught: "
                        f"{type(e).__name__}: {e}"
                    )

                    if attempt == max_attempts:
                        # ← BREAKPOINT D: all attempts exhausted → re-raise
                        logger.error(
                            f"[{func.__name__}] All {max_attempts} attempts failed."
                        )
                        raise   # ← re-raises the SAME exception with original traceback

                    logger.info(f"Retrying in {delay}s...")
                    time.sleep(delay)
                    delay *= backoff_factor  # 1s → 2s → 4s

        return wrapper
    return decorator


# ── Decorated Functions ────────────────────────────────────────

# Scenario 1: HTTPError from raise_for_status()
@retry(max_attempts=3, exceptions=(HTTPError,), backoff_factor=2)
def fetch_with_http_error(status_code: int):
    """
    httpbin.org/status/{code} always returns that status code.
    Use 500, 503, 429, 404 to trigger different HTTPErrors.
    """
    url = f"https://httpbin.org/status/{status_code}"
    logger.debug(f"GET {url}")

    response = requests.get(url, timeout=5)
    # ← BREAKPOINT E: response received, no exception yet
    # Inspect: response.status_code, response.ok, response.headers

    response.raise_for_status()
    # ← BREAKPOINT F: only reached if status < 400
    # If status >= 400 → raises HTTPError → jumps to except block in wrapper

    return response.json()


# Scenario 2: Timeout
@retry(max_attempts=3, exceptions=(Timeout,), backoff_factor=1)
def fetch_with_timeout():
    """
    httpbin.org/delay/10 waits 10s — our timeout=2 will trigger Timeout.
    Timeout is raised DURING the request, before any response is received.
    """
    response = requests.get("https://httpbin.org/delay/10", timeout=2)
    response.raise_for_status()
    return response.json()


# Scenario 3: Successful request
@retry(max_attempts=3, exceptions=(HTTPError, Timeout, ConnectionError))
def fetch_success():
    """Simple successful call — retry decorator does nothing special."""
    response = requests.get("https://httpbin.org/get", timeout=5)
    response.raise_for_status()
    return response.json()


# Scenario 4: Catching the re-raised exception at call site
def demo_catching_reraised_exception():
    """
    Shows that the exception propagates out of the decorator
    back to the caller with full traceback.
    """
    try:
        fetch_with_http_error(500)
    except HTTPError as e:
        # ← BREAKPOINT G: re-raised exception lands here
        logger.error(f"Caller caught HTTPError: {e}")
        logger.error(f"Status code:   {e.response.status_code}")
        logger.error(f"Response body: {e.response.text}")
        # e.response is the original Response object — always available on HTTPError


# ── Run All Scenarios ──────────────────────────────────────────
if __name__ == "__main__":

    print("\n" + "="*60)
    print("SCENARIO 1: HTTPError (500) — all retries fail")
    print("="*60)
    demo_catching_reraised_exception()

    print("\n" + "="*60)
    print("SCENARIO 2: Timeout — all retries fail")
    print("="*60)
    try:
        fetch_with_timeout()
    except Timeout as e:
        logger.error(f"Caller caught Timeout: {e}")

    print("\n" + "="*60)
    print("SCENARIO 3: Success — no retries needed")
    print("="*60)
    result = fetch_success()
    logger.info(f"Success! Origin IP: {result.get('origin')}")


SCENARIO 1: HTTPError (500) — all retries fail


2026-05-08 10:59:21,900 [DEBUG] [fetch_with_http_error] Attempt 1/3
2026-05-08 10:59:32,302 [DEBUG] GET https://httpbin.org/status/500
2026-05-08 10:59:35,387 [DEBUG] Starting new HTTPS connection (1): httpbin.org:443
2026-05-08 10:59:36,436 [DEBUG] https://httpbin.org:443 "GET /status/500 HTTP/1.1" 500 0
2026-05-08 11:00:14,043 [WARNING] [fetch_with_http_error] Attempt 1 caught: HTTPError: 500 Server Error: INTERNAL SERVER ERROR for url: https://httpbin.org/status/500
2026-05-08 11:00:25,912 [INFO] Retrying in 1s...
2026-05-08 11:01:01,940 [DEBUG] [fetch_with_http_error] Attempt 2/3
2026-05-08 11:02:01,211 [DEBUG] GET https://httpbin.org/status/500
2026-05-08 11:02:09,323 [DEBUG] Starting new HTTPS connection (1): httpbin.org:443
2026-05-08 11:02:10,113 [DEBUG] https://httpbin.org:443 "GET /status/500 HTTP/1.1" 500 0
2026-05-08 11:02:48,641 [WARNING] [fetch_with_http_error] Attempt 2 caught: HTTPError: 500 Server Error: INTERNAL SERVER ERROR for url: https://httpbin.org/status/500
202


SCENARIO 2: Timeout — all retries fail


2026-05-08 11:09:38,562 [DEBUG] [fetch_with_timeout] Attempt 1/3
2026-05-08 11:10:11,087 [DEBUG] Starting new HTTPS connection (1): httpbin.org:443
2026-05-08 11:11:07,422 [WARNING] [fetch_with_timeout] Attempt 1 caught: ReadTimeout: HTTPSConnectionPool(host='httpbin.org', port=443): Read timed out. (read timeout=2)
2026-05-08 11:11:10,752 [INFO] Retrying in 1s...
2026-05-08 11:11:28,810 [DEBUG] [fetch_with_timeout] Attempt 2/3
2026-05-08 11:11:52,873 [DEBUG] Starting new HTTPS connection (1): httpbin.org:443
2026-05-08 11:12:03,760 [WARNING] [fetch_with_timeout] Attempt 2 caught: ReadTimeout: HTTPSConnectionPool(host='httpbin.org', port=443): Read timed out. (read timeout=2)
2026-05-08 11:12:15,650 [INFO] Retrying in 1s...
2026-05-08 11:12:39,220 [DEBUG] [fetch_with_timeout] Attempt 3/3
2026-05-08 11:12:46,972 [DEBUG] Starting new HTTPS connection (1): httpbin.org:443
2026-05-08 11:12:55,730 [WARNING] [fetch_with_timeout] Attempt 3 caught: ReadTimeout: HTTPSConnectionPool(host='httpbi


SCENARIO 3: Success — no retries needed


2026-05-08 11:13:48,751 [DEBUG] [fetch_success] Attempt 1/3
2026-05-08 11:14:28,604 [DEBUG] Starting new HTTPS connection (1): httpbin.org:443
2026-05-08 11:14:29,364 [DEBUG] https://httpbin.org:443 "GET /get HTTP/1.1" 200 308
2026-05-08 11:15:00,682 [DEBUG] [fetch_success] Success on attempt 1
2026-05-08 11:15:27,361 [INFO] Success! Origin IP: 37.188.254.102


### Breakpoint Map for Your Debugger
```shell
Set breakpoints at these exact moments to watch the flow:

BREAKPOINT A → wrapper top of loop
               Watch: attempt number, args, kwargs

BREAKPOINT B → after result = func(...)
               Watch: result object, attempt number

BREAKPOINT C → inside except block
               Watch: e, type(e).__name__, e.args
               For HTTPError also: e.response.status_code

BREAKPOINT D → right before bare `raise`
               Watch: the full call stack — you'll see
               wrapper → fetch_with_http_error → raise_for_status

BREAKPOINT E → after requests.get(), before raise_for_status()
               Watch: response.status_code, response.ok
               This is where you SEE the 500 but NO exception yet

BREAKPOINT F → after raise_for_status()
               You'll NEVER hit this on a 500 —
               execution jumps straight to BREAKPOINT C

BREAKPOINT G → in caller's except block
               Watch: the exception is IDENTICAL to what was
               raised in raise_for_status() — same object, same traceback
```

---

### The 3 Exception Flows Side by Side
```shell
── Flow 1: HTTPError (raise_for_status) ──────────────────────

requests.get() → response(status=500) → raise_for_status()
                                              │
                                        HTTPError raised
                                              │
                              caught by except in wrapper (attempt 1)
                                              │
                                         sleep(1s)
                              caught by except in wrapper (attempt 2)
                                              │
                                         sleep(2s)
                              caught by except in wrapper (attempt 3)
                                              │
                                   attempt == max → bare raise
                                              │
                              HTTPError propagates to caller ✓


── Flow 2: Timeout ───────────────────────────────────────────

requests.get() → [waiting...2s] → Timeout raised
                                        │
                         NO response object ever created
                         (unlike HTTPError which has e.response)
                                        │
                         caught by except in wrapper → retry...


── Flow 3: Wrong exception type (not caught) ─────────────────

@retry(exceptions=(Timeout,))   ← only catches Timeout
def bad_fetch():
    response = requests.get(...)
    response.raise_for_status()  ← raises HTTPError

                    HTTPError raised
                         │
              except (Timeout,) as e: ← HTTPError is NOT Timeout
                         │
              NOT caught → bypasses retry entirely
                         │
              propagates directly to caller  ← silent retry failure!
```
This last flow is a **common interview trap** — always match your `exceptions=()` tuple precisely to what the function can actually raise.